# 01 Platform runtime smoke

Exercise callback retry, heartbeat, local JSONL replay and a bound terminal receipt.

This private `orchestrator_protected` notebook is generated from a reviewed Python template. It receives exact input versions and secrets through Kaggle runtime inputs; no credential is embedded in this notebook or written to its output.

In [ ]:
from __future__ import annotations

import hashlib
import os
import subprocess
import sys
from pathlib import Path

EXPECTED_SOURCE_SHA256 = '8fbc5444d058be867b762d2d8640780d61c188ea32bb37a2ae1034263ba5523b'
RUNTIME_CONTRACT = 'my-data-hub-platform-runtime-smoke.v1'
wheel = Path(os.environ.get('MY_DATA_HUB_WHEEL_PATH', ''))
if not wheel.is_file() or wheel.suffix != '.whl':
    raise RuntimeError('exact private my-data-hub wheel input is required')
expected_wheel_sha = os.environ.get('MY_DATA_HUB_WHEEL_SHA256', '')
if (len(expected_wheel_sha) != 64 or 
        hashlib.sha256(wheel.read_bytes()).hexdigest() != expected_wheel_sha):
    raise RuntimeError('my-data-hub wheel hash mismatch')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--no-deps', '--disable-pip-version-check', str(wheel)],
    check=True,
)

In [ ]:
PRIMARY_SOURCE = '"""Primary source for the private Kaggle platform/runtime smoke notebook."""\n\nfrom __future__ import annotations\n\nimport os\nfrom pathlib import Path\n\nfrom my_data_hub.hashing import canonical_json_bytes, sha256_value\nfrom my_data_hub.runtime_sdk import RuntimeClient, RuntimeEventType\n\n\ndef _required(name: str) -> str:\n    value = os.environ.get(name, "")\n    if not value:\n        raise RuntimeError(f"required runtime value is absent: {name}")\n    return value\n\n\ndef main() -> int:\n    output = Path("/kaggle/working")\n    client = RuntimeClient(\n        callback_url=_required("MY_DATA_HUB_CALLBACK_URL"),\n        run_secret=_required("MY_DATA_HUB_RUN_SECRET"),\n        run_id=_required("MY_DATA_HUB_RUN_ID"),\n        attempt_id=_required("MY_DATA_HUB_ATTEMPT_ID"),\n        service_instance_id=_required("MY_DATA_HUB_SERVICE_INSTANCE_ID"),\n        source_identity=_required("MY_DATA_HUB_SOURCE_IDENTITY"),\n        source_version=_required("MY_DATA_HUB_SOURCE_VERSION"),\n        epoch=int(_required("MY_DATA_HUB_EPOCH")),\n        spool_path=output / "runtime-events.jsonl",\n        heartbeat_interval_seconds=5.0,\n    )\n    client.replay_pending()\n    client.emit(RuntimeEventType.RUNTIME_STARTED, phase="smoke", status="running")\n    client.emit(RuntimeEventType.RUNTIME_HEARTBEAT, phase="smoke", status="healthy", data={"step": 1})\n    receipt = {\n        "schema_version": "my-data-hub-run-receipt.v1",\n        "task_run_id": client.run_id,\n        "provider_ref": _required("MY_DATA_HUB_SOURCE_IDENTITY"),\n        "source_version": _required("MY_DATA_HUB_SOURCE_VERSION"),\n        "source_sha256": _required("MY_DATA_HUB_SOURCE_SHA256"),\n        "terminal_state": "complete",\n        "output_sha256": sha256_value({"runtime_contract": "platform-runtime-smoke.v1"}),\n    }\n    (output / "my-data-hub-run-receipt.json").write_bytes(canonical_json_bytes(receipt))\n    client.emit(RuntimeEventType.RUNTIME_TERMINAL, phase="smoke", status="complete", data={"ok": True})\n    return 0\n'
if hashlib.sha256(PRIMARY_SOURCE.encode()).hexdigest() != EXPECTED_SOURCE_SHA256:
    raise RuntimeError('embedded primary source hash mismatch')
exec(compile(PRIMARY_SOURCE, '<my-data-hub-primary-source>', 'exec'), globals())

In [ ]:
raise SystemExit(globals()['main']())